# Meet Hugging Face (Week 10, in class)

By the end of this notebook you will have:
1. a Hugging Face account and a safely stored access token,
2. explored the Hub (models, datasets, Spaces, and model cards),
3. run a pre-trained model from Python with one line of code,
4. seen what a *token* is, and
5. built a small Streamlit app around the model.

**Tip:** run the setup cell below *first*. The first model download takes a few minutes.

In [ ]:
# Setup: run once (skip if already installed). In a terminal, drop the "!".
!pip install -q transformers torch

## Part 1: Create your Hugging Face account
1. Go to **huggingface.co** and click **Sign Up**.
2. Enter your email, a password, and choose a username. Finish any profile questions.
3. Open the confirmation email and click the link. (Check spam if it does not arrive.)
4. Click your profile picture (top right) and open your profile page.

**Check:** you can see your profile page with your username.

## Part 2: Access tokens (a password for your code)
An **access token** is a long secret string that lets *code* act as you on Hugging Face. You need one to download **gated or private** models, to **upload** your own models, and to avoid some download limits. All the models we use today are public, so you won't *need* a token, but you should know how to make and protect one.

**Create one:**
1. Click your profile picture, then **Settings**, then **Access Tokens**.
2. Click **Create new token**. Choose type **Read**. Give it a name like `cis3400-class`.
3. Click **Create** and **copy the token now**. It is shown only once.

**Rules for tokens:**
- ✅ Store it in an environment variable named `HF_TOKEN`, in Colab's **Secrets** panel (key icon on the left), or in Streamlit's `st.secrets`.
- ❌ Never paste it into a notebook cell, into `app.py`, into a file on GitHub, into a screenshot, or into a chat message.
- 🔁 If it ever leaks: go back to Settings, Access Tokens, **delete** that token, and create a new one.

**In Colab:** open the Secrets panel, click **Add new secret**, name it `HF_TOKEN`, paste the value, and turn on **Notebook access**. Then run the next cell. (Not using a token? Run it anyway; it will just say not found.)

In [ ]:
import os

# If you are in Colab and added an HF_TOKEN secret, copy it into an environment variable.
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass   # not in Colab, or no secret set: that's fine for public models

# Print only whether it exists. NEVER print the token itself.
print("Token found:", "HF_TOKEN" in os.environ)

In [ ]:
# Optional: confirm the token works by asking Hugging Face who you are.
from huggingface_hub import whoami

try:
    print("Logged in as:", whoami()["name"])
except Exception:
    print("No valid token found (fine for today).")

## Part 3: Explore the Hub (no code)
Open a new browser tab and work through these. Type your answers in the cell below.

1. Go to **huggingface.co/models**. On the left, under **Tasks**, choose **Text Classification**. Roughly how many models are there?
2. Sort by **Most downloads** (or Trending). What is the name of the top model?
3. Search for `distilbert-base-uncased-finetuned-sst-2-english` and open its **model card**. What *task* does it do, and what *labels* does it output?
4. What **dataset** was it trained on?
5. What **limitation or bias** does the card warn about?
6. What is the **license**? Then click **Files and versions**: how big is the model file?
7. Go to **huggingface.co/spaces** and search `sentiment`. Open any demo (a *Space*) and type a sarcastic sentence. Did it get it right?

**Vocabulary:** a **model** is a trained program; a **dataset** is data to train or test with; a **Space** is a live demo app; a **model card** is the model's documentation.

**My answers**

1. 
2. 
3. 
4. 
5. 
6. 
7. 

## Part 4: Your first pipeline
A Hugging Face **pipeline** bundles three steps into one line:
1. **Tokenize**: turn your text into numbers the model understands,
2. **Run the model**, and
3. **Clean up the output** into something readable.

Run the next cell. The first time, it downloads the model (a few hundred MB). A warning about a missing token is fine.

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
)

In [ ]:
classifier("I love this class!")

Notice the shape of the answer: a **list** containing a **dictionary** with a `label` and a `score`. The score is how *confident* the model is, not how *correct* it is.

**Your turn:** change the sentence above and run it again. Then try the three sentences below, and add your own: one positive, one **sarcastic**, and one **neutral** (a plain fact).

In [ ]:
my_sentences = [
    "The library has the best study rooms on campus.",
    "Oh great, another fire alarm at 2 a.m.",
    "The bookstore closes at 5 on weekdays.",
]

for s in my_sentences:
    r = classifier(s)[0]
    print(f"{r['label']:8} {r['score']:.3f}   {s}")

You can also pass a **whole list at once**. You get back one result per text, in the same order. (This is much faster than a loop for big lists, and you'll use it in the exercise.)

In [ ]:
results = classifier(my_sentences)
results

**Discuss:** Which sentence did the model get wrong, or seem unsure about? Notice this model has only two labels (positive and negative). What happens to the neutral sentence?

## Part 5: What is a token (the other meaning)?
You already met **access tokens**. In language models, a **token** is something different: a small chunk of text. Models don't read letters or words. They read tokens, each turned into a number.

- Common words are usually one token. Rare or long words get split into pieces (pieces that continue a word start with `##` in this model).
- A model can only read a limited number of tokens at once. For this model it is 512; longer text must be cut (`truncation=True`).
- Hosted language-model services often charge by the number of tokens used.

Every pipeline has its tokenizer built in. Let's look at it.

In [ ]:
tok = classifier.tokenizer

text = "Hugging Face is unbelievably awesome!"

tokens = tok.tokenize(text)
print("Tokens:", tokens)

ids = tok(text)["input_ids"]
print("IDs:   ", ids)
print("With special start/end tokens:", tok.convert_ids_to_tokens(ids))
print("Max tokens this model can read:", tok.model_max_length)

**Try it:** replace the text with a very long or unusual word (like `antidisestablishmentarianism`) and see how it splits.

## Part 6: Put it in an app
You now have everything for a small web app. We'll use **Streamlit**.

1. Create a folder named `week10-hf-analyzer` on your computer and open it in VS Code.
2. In it, create a file named `requirements.txt` containing these three lines:

```
streamlit
transformers
torch
```

3. Create a file named `app.py` with the code below.
4. In a terminal in that folder run: `pip install -r requirements.txt`, then `streamlit run app.py`.
5. Test it: type a comment, click **Analyze**, and try a sarcastic one.

```python
import streamlit as st
from transformers import pipeline

st.title("Campus Comment Sentiment Checker")


@st.cache_resource          # load the model once, not on every click
def load_model():
    return pipeline("sentiment-analysis",
                    model="distilbert-base-uncased-finetuned-sst-2-english")


classifier = load_model()

text = st.text_area("Enter a comment:")
if st.button("Analyze"):
    if not text.strip():
        st.warning("Please type a comment first.")
    else:
        result = classifier(text)[0]
        st.write("Label:", result["label"].lower())
        st.write("Score:", round(result["score"], 3))
```

Streamlit re-runs the whole file on every click. That is why the model is loaded inside a function with `@st.cache_resource`.

## Wrap-up
- **Hugging Face** is like GitHub for machine learning: models, datasets, and demo apps you can reuse.
- A **pipeline** runs a pre-trained model in one line.
- An **access token** is a secret key for your code. Keep it out of notebooks and GitHub.
- A **token** (in language models) is a chunk of text the model reads as a number.
- A confident score is **not** the same as a correct answer. That is the question for the exercise.